# 1 · RAG Augmentation — v5 (fixed)

**Fixes applied vs. the notebook this replaces** (full changelog at the bottom):

1. **A live API key was hard-coded in the previous version of this notebook** (`API_KEY = "gsk_..."`),
   overriding the `.env`-based `BACKEND_CONFIG` lookup that was already written directly above it.
   That key is embedded in plaintext in a notebook that may end up in a git repo, a shared drive, or
   a thesis submission bundle. **Treat that key as compromised — rotate/revoke it in the Groq
   console immediately if you haven't already.** This notebook now *only* reads keys from
   environment variables and fails fast with a clear error if the relevant one is missing.
2. `pip install dotenv` installs the wrong package — PyPI's `dotenv` is an unrelated, unmaintained
   namesake. The package actually imported (`from dotenv import load_dotenv`) comes from
   **`python-dotenv`**.
3. `df_augmented["response_status"]`, `row['thesis_label']`, and `r['nli_label']` were referenced
   in the save/spot-check cells but **never written** anywhere in the generation loop — the
   original notebook would crash on cell 10/11 after a full (rate-limited, possibly multi-day) run.
   All three columns are now populated per-row during generation.
4. Re-parses the DDI-2013 Test XML from scratch with its own filtering logic, duplicating
   (and silently diverging from) `0_ExtractData.ipynb`'s `parse_ddi_xml_dir`. This notebook now
   reads `test_raw_manifest.csv`, the file `0_ExtractData.ipynb` already produces for this exact
   purpose — one parser, one source of truth.
5. Hard-coded Windows path (`DDI_TEST_ROOT = r"DDICorpus\Test\..."`) with a commented-out Mac
   alternative. No longer needed at all once fix #4 removes the re-parsing step, but the general
   pattern is replaced with `os.path.join` / forward-slash paths for portability.
6. **New:** a deterministic, non-parser-dependent `entity_valid` check is computed for every
   generated response — literal case-insensitive substring presence of both `e1_text` and
   `e2_text` in the model's output. This directly implements §3.3 step 1 of the proposal
   ("Penyaringan Entitas") as a pre-filter rather than folding fake-drug detection into the
   learned NLI label space (see the `MAIN_NB` results review for why that mattered).
7. **New:** a final export cell reshapes the raw generation log into `holistic_test.csv`
   (`premise` / `hypothesis` / `label` / `scenario` / `entity_valid`, matching the schema
   `0_ExtractData.ipynb` now produces for train/val) — this glue step didn't exist anywhere in
   the original three notebooks; `holistic_test.csv` had to be assembled by hand out-of-band.


In [ ]:
# NOTE: install the correct package. `dotenv` (no prefix) on PyPI is a different,
# unmaintained project and does not provide `from dotenv import load_dotenv`.
%pip install -q python-dotenv


In [ ]:
import os
import time
import glob
import math

import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

load_dotenv()


## Backend switch

**Fix:** no hard-coded key. `API_KEY` is read exclusively from the environment variable named in
`BACKEND_CONFIG`, and the notebook raises immediately if it's unset — instead of silently running
with a stale/leaked key or failing deep inside `call_api` with a confusing auth error.


In [ ]:
BACKEND = "groq"   # <- "gemini" or "groq"

BACKEND_CONFIG = {
    "gemini": {
        "api_key_env": "GEMINI_API_KEY",
        "model":       "gemini-2.5-flash",
        "sleep_sec":   7,
        "rpm":         10,
        "rpd":         250,
    },
    "groq": {
        "api_key_env": "GROQ_API_KEY",
        "model":       "llama-3.3-70b-versatile",
        "sleep_sec":   5,
        "rpm":         30,
        "rpd":         1_000,
    },
}

cfg = BACKEND_CONFIG[BACKEND]

# FIX: no hard-coded fallback. Fail loudly and immediately if the key is missing,
# rather than silently using a leaked literal key checked into the notebook.
key_env_name = cfg["api_key_env"]
API_KEY = os.getenv(key_env_name)
if not API_KEY:
    raise EnvironmentError(
        f"Missing {key_env_name}. Set it in a local .env file "
        f"(never commit that file) before running this notebook.\n"
        f"  echo '{key_env_name}=your-key-here' >> .env"
    )

MODEL_NAME = cfg["model"]
SLEEP_SEC  = cfg["sleep_sec"]

TEST_MANIFEST_CSV = "test_raw_manifest.csv"   # produced by 0_ExtractData.ipynb

OUTPUT_CSV       = f"synthetic_rag_dataset_{BACKEND}.csv"
CHECKPOINT_CSV   = f"synthetic_rag_checkpoint_{BACKEND}.csv"
CHECKPOINT_EVERY = 20
MAX_RETRIES      = 4
FAKE_DRUG_NAME   = "Synthetozole"
SCENARIOS        = ["entailment", "contradiction", "neutral", "fake_drug"]

calls_per_pair       = len(SCENARIOS)
rpm, rpd             = cfg["rpm"], cfg["rpd"]
pairs_per_day        = rpd // calls_per_pair
wall_time_per_pair_s = calls_per_pair * SLEEP_SEC

print(f"Backend      : {BACKEND.upper()} / {MODEL_NAME}")
print(f"Rate limit   : {rpm} RPM  |  {rpd} RPD")
print(f"Sleep/call   : {SLEEP_SEC} s")
print(f"Wall time    : ~{wall_time_per_pair_s} s per pair")
print(f"Daily budget : ~{pairs_per_day} pairs/day  ({pairs_per_day * calls_per_pair} API calls)")
print(f"Output CSV   : {OUTPUT_CSV}")


In [ ]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types as genai_types
    gemini_client = genai.Client(api_key=API_KEY)
    print("Gemini client ready.")

elif BACKEND == "groq":
    from groq import Groq
    groq_client = Groq(api_key=API_KEY)
    print("Groq client ready.")


## Load the test manifest

**Fix:** reads `test_raw_manifest.csv` from `0_ExtractData.ipynb` instead of re-parsing the
DDI-2013 Test XML a second time with independent (and subtly different — e.g. this notebook used
to additionally require non-empty `e1_text`/`e2_text` strings, a filter `0_ExtractData.ipynb`
doesn't apply) filtering logic. One parser, one source of truth, no risk of the two notebooks'
row counts silently drifting apart.


In [ ]:
df_test = pd.read_csv(TEST_MANIFEST_CSV)

# Same non-empty-entity guard the old inline parser applied, now applied explicitly
# and visibly here rather than being buried inside a duplicate XML parser.
df_test = df_test[
    df_test["e1_text"].notna() & (df_test["e1_text"].str.strip() != "") &
    df_test["e2_text"].notna() & (df_test["e2_text"].str.strip() != "")
].reset_index(drop=True)

print(f"Total pairs        : {len(df_test):,}")
print(f"Unique premises    : {df_test['premise'].nunique():,}")
print()
print("Interaction type distribution:")
print(df_test["interaction_type"].value_counts().to_string())
print()

total_calls = len(df_test) * calls_per_pair
days_needed = math.ceil(total_calls / cfg["rpd"])
print(f"Total API calls needed : {total_calls:,}")
if days_needed > 1:
    print(f"[INFO] Exceeds 1-day RPD budget. Estimated run time: {days_needed} day(s).")
    print("       Checkpoint saves every 20 pairs — safe to stop and resume.")


## Prompts

Unchanged from the previous version.

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical pharmacology assistant generating concise summaries "
    "of drug-drug interaction (DDI) information for a medical literature review system. "
    "Write in 5-6 sentences. Do not add disclaimers, safety warnings, or meta-commentary "
    "about the nature of the task. Output the summary only."
)


def build_prompts(context: str, e1: str, e2: str) -> dict:
    return {
        "entailment": (
            f"Summarise the following drug-drug interaction finding accurately and faithfully. "
            f"Preserve the clinical direction of the effect (e.g. increases, decreases, "
            f"no significant change).\n\nContext: {context}"
        ),
        "contradiction": (
            f"You are writing a summary that reflects an alternative clinical finding "
            f"that directly contradicts the source text. "
            f"If the source states no significant interaction exists between {e1} and {e2}, "
            f"your summary must state that a clinically significant interaction does exist. "
            f"If the source states an interaction exists (e.g. one drug increases or decreases "
            f"the levels of the other), your summary must state the opposite effect or that "
            f"no interaction occurs. Keep both drug names ({e1} and {e2}) in the summary. "
            f"Do not add hedging phrases like 'however' or 'contrary to some reports'. "
            f"State the contradicting finding as clinical fact.\n\nContext: {context}"
        ),
        "neutral": (
            f"Write a brief description of {e1} and {e2} that mentions both drugs by name "
            f"but discusses ONLY their pharmacological drug class or general mechanism of action. "
            f"Do NOT mention, imply, or reference any interaction, combined effect, "
            f"pharmacokinetic change, clinical outcome, or safety profile between them. "
            f"The output must be clinically irrelevant to whether these drugs interact.\n\nContext: {context}"
        ),
        "fake_drug": (
            f"Summarise the following drug-drug interaction finding accurately, "
            f"but replace every mention of '{e2}' with the drug name '{FAKE_DRUG_NAME}'. "
            f"Keep all other clinical details and drug names ({e1}) unchanged. "
            f"Do not add any explanation of the substitution.\n\nContext: {context}"
        ),
    }


print("Prompt builder defined.")


## API helpers

**New:** `check_entity_validity()` — a deterministic, spaCy-independent check that both
`e1_text` and `e2_text` are literally present (case-insensitively) in the generated text. For the
`fake_drug` scenario this will (by construction) come back `False`, since `e2_text` was
deliberately replaced by `FAKE_DRUG_NAME`. This is computed here, at generation time, so it's
available immediately rather than needing to be reverse-engineered later from whether the
downstream SVO parser happened to find a triplet (which conflates *parser failure* with
*genuine hallucinated-entity detection* — see `2_Atomisation.ipynb`'s changelog).


In [ ]:
def call_api(prompt: str, retries: int = MAX_RETRIES) -> str:
    """Send a prompt to the configured backend and return the response text.
    Retries with exponential backoff on any exception. Sleeps SLEEP_SEC after
    every successful call to respect the free-tier RPM.
    """
    for attempt in range(retries):
        try:
            if BACKEND == "gemini":
                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=genai_types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0.7,
                        max_output_tokens=512,
                    ),
                )
                text = response.text.strip()

            elif BACKEND == "groq":
                response = groq_client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    temperature=0.7,
                    max_tokens=512,
                )
                text = response.choices[0].message.content.strip()

            time.sleep(SLEEP_SEC)
            return text

        except Exception as e:
            wait = SLEEP_SEC * (2 ** attempt)
            print(f"    [Attempt {attempt+1}/{retries}] {type(e).__name__}: {e} - retry in {wait}s")
            time.sleep(wait)

    print(f"    [FAILED] All {retries} attempts exhausted.")
    return ""


def validate_response(text: str) -> str:
    """Flag empty, refused, or suspiciously short generations."""
    if not text:
        return "empty"
    refusal_markers = [
        "i cannot", "i'm unable", "i am unable",
        "as an ai", "i can't assist", "i can't provide",
        "this request", "harmful content",
    ]
    if any(m in text.lower() for m in refusal_markers):
        return "refused"
    if len(text.split()) < 15:
        return "too_short"
    return "ok"


def check_entity_validity(text: str, e1: str, e2: str) -> bool:
    """Deterministic OOV/hallucinated-entity check: both known drug names must
    literally appear in the generated text. Independent of any downstream
    NLP parser, so it can't be confounded by parser failure. This is the
    §3.3 "Penyaringan Entitas" pre-filter, applied at generation time.
    """
    if not text:
        return False
    t = text.lower()
    return (str(e1).lower().strip() in t) and (str(e2).lower().strip() in t)


print("API helpers defined.")


## Main generation loop

**Fix:** `response_status`, `thesis_label`, `nli_label`, and `entity_valid` are now written into
every row at generation time (previously referenced downstream but never populated — a guaranteed
crash on a real, multi-day, rate-limited run). `nli_label` uses the 3-way mapping
(`fake_drug -> neutral`); the raw `scenario` (which prompt template produced this row) is kept as
a separate column for diagnostics/per-scenario accuracy slicing, exactly as `MAIN_NB`'s
`groupby('scenario')` already expects — it should not be reused as the training target.


In [ ]:
THESIS_LABEL = {
    "entailment":    "skenario_A",
    "contradiction": "skenario_B",
    "neutral":       "skenario_C",
    "fake_drug":     "skenario_D",
}
NLI_LABEL = {
    "entailment":    "entailment",
    "contradiction": "contradiction",
    "neutral":       "neutral",
    "fake_drug":     "neutral",   # OOV-filtered; not a semantic-inference class
}

if os.path.exists(CHECKPOINT_CSV):
    df_existing    = pd.read_csv(CHECKPOINT_CSV)
    done_ids       = set(df_existing["original_id"].unique())
    augmented_data = df_existing.to_dict("records")
    print(f"Resuming - {len(done_ids)} pairs already done ({len(augmented_data)} records).")
else:
    done_ids       = set()
    augmented_data = []
    print("No checkpoint - starting fresh.")

rows_processed = 0

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Pairs"):
    pair_id = row["original_id"]
    if pair_id in done_ids:
        continue

    context = row["premise"]
    e1      = row["e1_text"]
    e2      = row["e2_text"]
    prompts = build_prompts(context, e1, e2)

    for scenario in SCENARIOS:
        synthetic_text = call_api(prompts[scenario])
        status         = validate_response(synthetic_text)

        if status != "ok":
            print(f"  [WARN] {pair_id} | {scenario} | {status}")

        # For fake_drug, entity validity is checked against the ORIGINAL e2 —
        # a valid substitution should make e2 disappear from the text, so
        # entity_valid is expected to be False here by construction.
        entity_valid = check_entity_validity(synthetic_text, e1, e2)

        augmented_data.append({
            "original_id":          pair_id,
            "source":               row.get("source_domain", ""),
            "e1_text":              e1,
            "e2_text":              e2,
            "premise":              context,
            "synthetic_rag_output": synthetic_text,
            "source_interaction_type": row["interaction_type"],  # provenance only, not the NLI label
            "scenario":             scenario,
            "thesis_label":         THESIS_LABEL[scenario],
            "nli_label":            NLI_LABEL[scenario],
            "entity_valid":         entity_valid,
            "response_status":      status,
            "model":                MODEL_NAME,
        })

    done_ids.add(pair_id)
    rows_processed += 1

    if rows_processed % CHECKPOINT_EVERY == 0:
        pd.DataFrame(augmented_data).to_csv(CHECKPOINT_CSV, index=False)
        print(f"  [Checkpoint] {len(augmented_data)} records after {rows_processed} pairs.")

print(f"\nDone. Total records: {len(augmented_data):,}")


## Save and quality report

In [ ]:
df_augmented = pd.DataFrame(augmented_data)
df_augmented.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_augmented):,} rows to {OUTPUT_CSV}")
print()
print("Scenario x Source distribution:")
print(df_augmented.groupby(["source", "scenario"]).size().unstack(fill_value=0).to_string())
print()
print("Response status breakdown:")
print(df_augmented.groupby(["scenario", "response_status"]).size().to_string())
print()
print("Entity-validity breakdown (expect ~all True except fake_drug):")
print(df_augmented.groupby(["scenario", "entity_valid"]).size().unstack(fill_value=0).to_string())

flagged = df_augmented[df_augmented["response_status"] != "ok"]
if len(flagged):
    print(f"\n{len(flagged)} flagged responses need review:")
    print(flagged[["original_id", "scenario", "response_status"]].to_string())

if os.path.exists(CHECKPOINT_CSV):
    os.remove(CHECKPOINT_CSV)
    print("\nCheckpoint removed.")


In [ ]:
# ── Spot-check: all 4 scenarios for one pair ─────────────────────────────────
sample_id = df_augmented["original_id"].iloc[0]
sample    = df_augmented[df_augmented["original_id"] == sample_id]

print(f"=== Source (id={sample_id}) ===")
print("Premise       :", sample.iloc[0]["premise"])
print("e1 / e2       :", sample.iloc[0]["e1_text"], "/", sample.iloc[0]["e2_text"])
print("Source itype  :", sample.iloc[0]["source_interaction_type"])
print()
for _, r in sample.iterrows():
    print(f"--- [{r['thesis_label']}] {r['scenario']} (nli={r['nli_label']}, entity_valid={r['entity_valid']}) ---")
    print(r["synthetic_rag_output"])
    print()


## Export `holistic_test.csv`

**New cell.** This glue step didn't exist in the original three notebooks — `holistic_test.csv`
had to be hand-built out-of-band to match the schema `MAIN_NB.NLIDataBundle(prefix="holistic")`
expects. It's added here, immediately after generation, so the whole pipeline runs end-to-end
without a manual reshaping step.

Column mapping: `premise` -> `premise` (unchanged), `synthetic_rag_output` -> `hypothesis`,
`nli_label` -> `label` (3-way), `scenario` kept as-is for diagnostics, `entity_valid` carried
through unchanged. `original_id` is suffixed with the scenario so all four generations of the
same underlying DDI pair get distinct, traceable rows — exactly mirroring how `original_id` is
constructed for the holistic *train* set in `0_ExtractData.ipynb`.


In [ ]:
os.makedirs("data", exist_ok=True)

holistic_test_df = df_augmented[[
    "original_id", "e1_text", "e2_text", "premise", "synthetic_rag_output",
    "nli_label", "scenario", "entity_valid",
]].rename(columns={"synthetic_rag_output": "hypothesis", "nli_label": "label"}).copy()

holistic_test_df["original_id"] = (
    holistic_test_df["original_id"].astype(str) + "__" + holistic_test_df["scenario"]
)

holistic_test_df.to_csv("data/holistic_test.csv", index=False)
print(f"Saved data/holistic_test.csv - {len(holistic_test_df):,} rows")
print("\nLabel distribution:")
print(holistic_test_df["label"].value_counts())
print("\nScenario distribution:")
print(holistic_test_df["scenario"].value_counts())


## Changelog

| # | Issue in original notebook | Fix |
|---|---|---|
| 1 | Hard-coded live API key in plaintext, overriding the `.env`-based config right above it | Removed; reads exclusively from env var, fails fast if missing. **Rotate the leaked key.** |
| 2 | `pip install dotenv` installs the wrong PyPI package | `pip install python-dotenv` |
| 3 | `response_status`, `thesis_label`, `nli_label` referenced downstream but never written during generation -> guaranteed crash after a long rate-limited run | All four fields (plus new `entity_valid`) populated per-row in the main loop |
| 4 | Re-parses DDI-2013 Test XML independently of `0_ExtractData.ipynb`, with silently divergent filtering | Reads `test_raw_manifest.csv` produced by `0_ExtractData.ipynb` — single source of truth |
| 5 | Hard-coded Windows path with a commented-out Mac alternative | Removed (no longer parses XML here at all) |
| 6 | Fake-drug/hallucination detection had no explicit, parser-independent signal | Added deterministic `check_entity_validity()` at generation time |
| 7 | No step produced `holistic_test.csv` in the schema `MAIN_NB` needs | Added export cell (`premise`/`hypothesis`/`label`/`scenario`/`entity_valid`) |
